In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker V3 Model Optimization Example

This notebook demonstrates how to use SageMaker V3 ModelBuilder to optimize a JumpStart model for improved inference performance.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.

In [2]:
# Import required libraries
import json
import uuid
import time
import boto3

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.core.resources import EndpointConfig
from sagemaker.core.helper.session_helper import Session

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Step 1: Configure Model and Session

We'll optimize a Llama 3 model from JumpStart using AWQ quantization.

In [3]:
# Configuration
MODEL_ID = "meta-textgeneration-llama-3-8b-instruct"
MODEL_NAME_PREFIX = "jumpstart-optimize-example"
ENDPOINT_NAME_PREFIX = "jumpstart-optimize-example-endpoint"
session = Session()
AWS_ACCOUNT_ID = session.account_id()
AWS_REGION = session.boto_region_name

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"
job_name = f"js-optimize-{int(time.time())}"

print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")
print(f"Optimization job name: {job_name}")

Model name: jumpstart-optimize-example-a01222a5
Endpoint name: jumpstart-optimize-example-endpoint-a01222a5
Optimization job name: js-optimize-1784532917


## Step 2: Create Schema Builder

Define the input/output schema for the text generation model.

In [4]:
# Create schema builder for text generation
sample_input = {"inputs": "What are falcons?", "parameters": {"max_new_tokens": 32}}
sample_output = [{"generated_text": "Falcons are small to medium-sized birds of prey."}]

schema_builder = SchemaBuilder(sample_input, sample_output)
print("Schema builder created successfully!")

Schema builder created successfully!


## Step 3: Initialize SageMaker Session

Create a SageMaker session with the specified AWS region.

In [5]:
# Create SageMaker session
boto_session = boto3.Session(region_name=AWS_REGION)
sagemaker_session = Session(boto_session=boto_session)
print(f"SageMaker session created for region: {AWS_REGION}")

SageMaker session created for region: us-west-2


## Step 4: Create ModelBuilder

Initialize the ModelBuilder with the JumpStart model ID and schema.

In [6]:
# Initialize ModelBuilder
model_builder = ModelBuilder(
    model=MODEL_ID,
    schema_builder=schema_builder,
    sagemaker_session=sagemaker_session,
)
print("ModelBuilder created successfully!")

[07/20/26 07:35:18] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=308737;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=308738;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

Model 'meta-textgeneration-llama-3-8b-instruct' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMetadata/eula/llama3Eula.txt for terms of use.


                    INFO     Model 'meta-textgeneration-llama-3-8b-instruct' requires accepting        ]8;id=308745;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/utils.py\utils.py]8;;\:]8;id=308746;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/utils.py#647\647]8;;\
                             end-user license agreement (EULA). See                                                
                             https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMeta             
                             data/eula/llama3Eula.txt for terms of use.                                            

Using model 'meta-textgeneration-llama-3-8b-instruct' with wildcard version identifier '*'. You can pin to version '2.25.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


                    WARNING  Using model 'meta-textgeneration-llama-3-8b-instruct' with wildcard       ]8;id=308753;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/cache.py\cache.py]8;;\:]8;id=308754;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/cache.py#624\624]8;;\
                             version identifier '*'. You can pin to version '2.25.0' for more stable               
                             results. Note that models may have different input/output signatures                  
                             after a major version upgrade.                                                        

                    DEBUG    JumpStart Model ID detected.                               ]8;id=308760;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=308761;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#2914\2914]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=308767;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=308768;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=308775;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=308776;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

[07/20/26 07:35:19] WARNING  Could not verify permissions for role                         ]8;id=308782;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=308783;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

ModelBuilder created successfully!


## Step 5: Optimize the Model

Optimize the model using AWQ quantization for improved inference performance. This step may take up to 30 minutes to complete!

In [7]:
# Optimize the model with AWQ quantization
print("Optimizing JumpStart model...")
optimized_model = model_builder.optimize(
    instance_type="ml.g5.2xlarge",
    image_uri="763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124",
    output_path=f"s3://notebook-test-engine-ds-674622101542-usw2/optimize-output/jumpstart-{unique_id}/",
    quantization_config={"OverrideEnvironment": {"OPTION_QUANTIZE": "awq"}},
    accept_eula=True,
    job_name=job_name,
    model_name=model_name,
)
print(f"Model Successfully Optimized: {optimized_model.model_name}")

Optimizing JumpStart model...


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=308790;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=308791;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:35:20] DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=308797;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=308798;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#1380\1380]8;;\
                             is not handling MLflow model input                                                    

                    DEBUG    Building for JumpStart model ID...                               ]8;id=308805;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=308806;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#2786\2786]8;;\

Model 'meta-textgeneration-llama-3-8b-instruct' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMetadata/eula/llama3Eula.txt for terms of use.


...

[07/20/26 08:00:31] INFO     Creating model with name: jumpstart-optimize-example-a01222a5   ]8;id=308869;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=308870;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1922\1922]8;;\

[07/20/26 08:00:32] DEBUG    No boto3 session provided. Creating a new session.                        ]8;id=308877;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=308878;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#357\357]8;;\

                    DEBUG    No config provided. Using default config.                                 ]8;id=308884;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=308885;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#365\365]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=308891;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=308892;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

Model Successfully Optimized: jumpstart-optimize-example-a01222a5


## Step 6: Deploy the Optimized Model

Deploy the optimized model to a SageMaker endpoint for real-time inference.

In [8]:
# Deploy the optimized model to an endpoint
print("Deploying optimized model to endpoint...")
core_endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1
)
print(f"Endpoint Successfully Created: {core_endpoint.endpoint_name}")

Deploying optimized model to endpoint...


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=308897;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=308898;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 08:00:33] INFO     Creating endpoint-config with name                              ]8;id=308904;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=308905;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1093\1093]8;;\
                             jumpstart-optimize-example-endpoint-a01222a5                                          

                    INFO     Creating endpoint with name                                     ]8;id=308911;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=308912;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1125\1125]8;;\
                             jumpstart-optimize-example-endpoint-a01222a5                                          

[07/20/26 08:00:34] WARNING  Failed to enable live logging: An error occurred                ]8;id=308918;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=308919;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#2844\2844]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::674622101542:assumed-role/NotebookTestEngine-Proce                       
                             ssingJobRole/SageMaker is not authorized to perform:                                  
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-west-2:674622101542:log-group:/aws/sagemaker/En                       
                             dpoints/jumpstart-optimize-example-endpoint-a01222a5:log-stream                       
                             : because no identity-based policy allows the                                         
                             logs:FilterLogEvents action. Fallback to default logging...                           

/usr/local/lib/python3.12/dist-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[07/20/26 08:08:05] INFO     ✅ Deployment successful: Endpoint                               ]8;id=308925;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=308926;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#2977\2977]8;;\
                             'jumpstart-optimize-example-endpoint-a01222a5' using DJL_SERVING                      
                             in SAGEMAKER_ENDPOINT mode (ARN:                                                      
                             arn:aws:sagemaker:us-west-2:674622101542:endpoint/jumpstart-opti                      
                             mize-example-endpoint-a01222a5)                                                       

Endpoint Successfully Created: jumpstart-optimize-example-endpoint-a01222a5


## Step 7: Test the Optimized Endpoint

Send a test request to verify the optimized model is working correctly.

In [9]:
# Test optimized model invocation
test_data = {
    "inputs": "What are the benefits of machine learning?",
    "parameters": {"max_new_tokens": 50}
}

result = core_endpoint.invoke(
    body=json.dumps(test_data),
    content_type="application/json"
)

response_body = result.body.read().decode('utf-8')
prediction = json.loads(response_body)
print(f"Result of invoking optimized endpoint: {prediction}")

Result of invoking optimized endpoint: {'generated_text': ' Machine learning has numerous benefits, including:\n1. Improved accuracy: Machine learning algorithms can learn from data and improve their accuracy over time, reducing the risk of human error.\n2. Increased efficiency: Machine learning can automate many tasks, freeing up human resources'}


## Step 8: Clean Up Resources

Clean up the created resources to avoid ongoing charges.

In [10]:
# Clean up resources
core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)

# Delete in the correct order
optimized_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("Optimized model and endpoint successfully deleted!")

[07/20/26 08:08:07] INFO     Deleting Model - jumpstart-optimize-example-a01222a5                ]8;id=308933;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=308934;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

                    INFO     Deleting Endpoint - jumpstart-optimize-example-endpoint-a01222a5    ]8;id=308940;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=308941;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10428\10428]8;;\

[07/20/26 08:08:08] INFO     Deleting EndpointConfig -                                           ]8;id=308947;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=308948;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#11220\11220]8;;\
                             jumpstart-optimize-example-endpoint-a01222a5                                          

Optimized model and endpoint successfully deleted!


## Summary

This notebook demonstrated:
1. Creating a ModelBuilder with a JumpStart model
2. Optimizing the model using AWQ quantization
3. Deploying the optimized model to a SageMaker endpoint
4. Making inference requests to the optimized endpoint
5. Cleaning up resources

The V3 ModelBuilder's optimize() method makes it easy to improve model performance with quantization and other optimization techniques!